In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
HN_COMMENTS_CSV = '../dataset/hn_gh_ai_comment_sentiment.csv'
GROUND_TRUTH_CSV = '../dataset/ground_truth_hn_ai_comment.csv'

NUM_COMMENTS = 354

In [3]:
data = pd.read_csv(HN_COMMENTS_CSV)
ground_truth = pd.read_csv(GROUND_TRUTH_CSV)

In [4]:
comment_df = data[~data['comment_id'].isin(ground_truth['comment_id'])].copy()
print(f"Total comments available for sampling: {len(comment_df)}")
print(f"Total stories in comment_df: {len(comment_df['story_id'].unique())}")

Total comments available for sampling: 4455
Total stories in comment_df: 1002


In [5]:
comment_df['word_count'] = comment_df['comment_text'].apply(lambda x: len(str(x).split()))

In [6]:
comment_df.head(1)

,comment_id,comment_text,comment_datetime,story_id,story_title,url,story_datetime,comment_sentiment,comment_sentiment_reason,story_sentiment,story_sentiment_reason,word_count
0,34303291,You: What is Arc?Paul: Arc is a new dialect of...,2023-01-08 21:24:08 UTC,34286739,Show HN: An AI Clone of Paul Graham,https://github.com/louis030195/paul-graham-clone,2023-01-07 10:00:00 UTC,0.0,The comment provides a factual description of ...,0,The headline presents an AI clone of a notable...,114


In [7]:
def calculate_dynamic_targets(df_counts, total_target=354, base_target=39):
    """
    df_counts: A dictionary or Pandas Series of the total available comments per combination.
    """
    targets = {}
    
    # 1. Base shortfall
    deficit = total_target - (base_target * len(df_counts))
    surplus_combos = []
    
    # 2. Assign base targets and find the real deficit
    for combo, count in df_counts.items():
        if count <= base_target:
            targets[combo] = count
            deficit += (base_target - count) # Add the missing amount to our deficit
        else:
            targets[combo] = base_target
            surplus_combos.append(combo)

    # 3. Distribute the deficit proportionally to the larger groups
    if deficit > 0 and surplus_combos:
        # Get the total population of the groups that can take more comments
        total_surplus_population = sum(df_counts[c] for c in surplus_combos)
        
        additions = {}
        for combo in surplus_combos:
            # Calculate proportion of the burden this combo should take
            proportion = df_counts[combo] / total_surplus_population
            additions[combo] = int(deficit * proportion)
            
        # Add the proportional amounts
        for combo, add in additions.items():
            targets[combo] += add
            
        # 4. Handle rounding errors (integer math might leave a remainder)
        remainder = deficit - sum(additions.values())
        if remainder > 0:
            # Sort the big combos by their original size, largest first
            sorted_surplus = sorted(surplus_combos, key=lambda k: df_counts[k], reverse=True)
            
            # Give one extra comment to the largest buckets until remainder is 0
            for i in range(remainder):
                targets[sorted_surplus[i]] += 1
                
    return targets

In [8]:
# 1. Get the actual counts for each combination of story_sentiment and comment_sentiment
counts_series = comment_df.groupby(['story_sentiment', 'comment_sentiment']).size()
current_counts = counts_series.to_dict()

# 2. Generate the exact targets
targets = calculate_dynamic_targets(current_counts, total_target=354, base_target=39)

print("Target samples per combination:")
for k, v in targets.items():
    print(f"{k}: {v}")

# 3. Sample the comments based on the targets
df_shuffled = comment_df.sample(frac=1, random_state=42).reset_index(drop=True)
selected_indices = []
story_counts = {}

for index, row in df_shuffled.iterrows():
    if sum(targets.values()) == 0:
        break
        
    combo = (row['story_sentiment'], row['comment_sentiment'])
    story_id = row['story_id']
        
    if targets[combo] > 0:
        targets[combo] -= 1
        story_counts[story_id] = story_counts.get(story_id, 0) + 1
        selected_indices.append(index)

sampled_df = df_shuffled.loc[selected_indices].copy()

Target samples per combination:
(-1, -1.0): 21
(-1, 0.0): 39
(-1, 1.0): 14
(0, -1.0): 42
(0, 0.0): 62
(0, 1.0): 52
(1, -1.0): 39
(1, 0.0): 43
(1, 1.0): 42


In [9]:
# Verify the results
print(f"Total comments sampled: {len(sampled_df)}")
print(f"Total stories sampled: {len(sampled_df['story_id'].unique())}")
print(f"Max comments per story: {sampled_df['story_id'].value_counts().max()}")

print("\nShort comments (<= 4 words) per combination:")
short_df = sampled_df[sampled_df['word_count'] <= 4]
print(short_df.groupby(['story_sentiment', 'comment_sentiment']).size())

print("\nLong comments (>= 5 words) per combination:")
long_df = sampled_df[sampled_df['word_count'] >= 5]
print(long_df.groupby(['story_sentiment', 'comment_sentiment']).size())

print("\nAll comments per combination:")
print(sampled_df.groupby(['story_sentiment', 'comment_sentiment']).size())

Total comments sampled: 354
Total stories sampled: 190
Max comments per story: 23

Short comments (<= 4 words) per combination:
story_sentiment  comment_sentiment
-1               -1.0                 1
                  1.0                 1
 0               -1.0                 1
                  0.0                 2
                  1.0                 2
 1                0.0                 1
                  1.0                 1
dtype: int64

Long comments (>= 5 words) per combination:
story_sentiment  comment_sentiment
-1               -1.0                 20
                  0.0                 39
                  1.0                 13
 0               -1.0                 41
                  0.0                 60
                  1.0                 50
 1               -1.0                 39
                  0.0                 42
                  1.0                 41
dtype: int64

All comments per combination:
story_sentiment  comment_sentiment
-1              

In [10]:
sampled_df.to_csv('../dataset/sampled_354_hn_gh_ai_comment.csv', index=False)